# QC: anatomy → reference brain, one experiment

Checks the 3D registration of each fish's anatomy onto the reference brain (`ref_L433_f02`), one experiment at a time.
For every fish it saves a figure with 5 depths (reference | fish | overlay) and a score; for the experiment it saves an
overview of all fish at the same reference depths and a table where you write your decision per fish.
Outputs go to `Paper/2p_derived/registration/<experiment>/reg_to_ref/`.

## 1. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Walk up to the repo root (marked by utils.py) so the project modules import from anywhere.
repo_root = Path.cwd()
while not (repo_root / "utils.py").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from registration.experiment_fish import find_experiment_fish_folders, list_experiment_fish
from registration.image_utils import norm01
from registration.qc_plotting import save_figure
from registration.reference_qc import (
    add_decision_columns,
    common_depths,
    downsample_stack,
    fish_mean_image,
    read_nrrd_stack,
    read_previous_decisions,
    reference_anatomy_path,
    scale_fish_image,
    spread_depths,
    summarize_fish,
    warped_anatomy_path,
    write_qc_table,
)

## 2. Settings
Choose the experiment here.

In [ ]:
EXPERIMENT = "Exp_9_RF_memory_inhibition"  # Exp_1_flickering, Exp_5_map_positions, Exp_7_trans_stat, Exp_8_accumulation_ev

LAB_ROOT = Path(r"C:\Users\suribear\OneDrive - Université de Lausanne\Lab")
DATA_ROOT = LAB_ROOT / "Data" / "2p"  # one folder per fish, named "<fish_id>..."
OVERVIEW_CSV = LAB_ROOT / "Results" / "Paper" / "2p_derived" / "global" / "overview.csv"
QC_DIR = LAB_ROOT / "Results" / "Paper" / "2p_derived" / "registration" / EXPERIMENT / "reg_to_ref"
TABLE_PATH = QC_DIR / f"{EXPERIMENT}_ref_qc.csv"
OVERVIEW_PATH = QC_DIR / f"{EXPERIMENT}_ref_overview.png"

REFERENCE_NAME = "ref_L433_f02"
REFERENCE_FISH_ID = "L433_f02"
REFERENCE_FOLDER = DATA_ROOT / "L433_f02_Exp_1_flickering"
Z_SPACING_UM = 2.0  # reference slice spacing (NRRD header)

N_DEPTHS = 5  # depths shown per fish and in the overview
FISH_FIGURE_DOWNSAMPLE = 2  # 750 px -> 375 px per panel
OVERVIEW_DOWNSAMPLE = 4  # 750 px -> 188 px per panel
CROP_MARGIN_PX = 10  # margin around the fish's extent, in downsampled pixels
REFERENCE_CLIP_PERCENTILES = (1, 99.5)  # reference contrast range
FISH_CLIP_PERCENTILES = (10, 99.5)  # fish contrast range, over its data pixels only
QC_DPI = 100  # matches the downsampled panel size, keeps files ~1-2 MB
PILOT_CORRELATION = 0.70  # L978_f01, checked by eye: a typical good registration
LOW_CORRELATION = 0.5  # good fish score ~0.6-0.7; a flip or depth shift falls well below this

## 3. Reference brain
Loaded once and kept for all fish.

In [ ]:
reference_stack = read_nrrd_stack(reference_anatomy_path(REFERENCE_FOLDER))
reference_thumbnail = downsample_stack(reference_stack, OVERVIEW_DOWNSAMPLE)
print(f"Reference: {reference_stack.shape} (z, y, x)")

## 4. Which fish have a registration
The experiment's fish from the Paper overview (without `exclude`), and whether their warped anatomy exists.

In [ ]:
experiment_fish = list_experiment_fish(OVERVIEW_CSV, EXPERIMENT)
fish_folders = find_experiment_fish_folders(DATA_ROOT, experiment_fish)
status_table = pd.DataFrame([
    {"fish_id": fish["fish_id"],
     "overview_decision": fish["final_decision"],
     "folder": fish_folders[fish["fish_id"]].name if fish_folders[fish["fish_id"]] else None,
     "registered": bool(fish_folders[fish["fish_id"]]) and warped_anatomy_path(fish_folders[fish["fish_id"]], fish["fish_id"], REFERENCE_NAME).exists()}
    for fish in experiment_fish
])
status_table

## 5. Figure layouts
Colours, sizes and crops of the QC figures; edit them here.

In [ ]:
def overlay_rgb(reference_image, fish_image):
    """Combine two 0-1 images: reference in magenta, fish in green (white where both are bright).

    Args:
        reference_image (numpy.ndarray): 2D image scaled to 0-1.
        fish_image (numpy.ndarray): 2D image scaled to 0-1, same shape.

    Returns:
        numpy.ndarray: (y, x, 3) RGB image.
    """
    rgb_image = np.stack([reference_image, fish_image, reference_image], axis=-1)
    return rgb_image


def crop_slices(bounding_box, downsample_factor, margin_px):
    """Turn a full-resolution (y0, y1, x0, x1) box into slices of a downsampled image.

    Args:
        bounding_box (tuple): (y0, y1, x0, x1) at full resolution, end-exclusive.
        downsample_factor (int): How much the image was shrunk.
        margin_px (int): Extra pixels on every side, in downsampled pixels.

    Returns:
        tuple: (row slice, column slice).
    """
    y0, y1, x0, x1 = bounding_box
    row_slice = slice(max(y0 // downsample_factor - margin_px, 0), -(-y1 // downsample_factor) + margin_px)
    column_slice = slice(max(x0 // downsample_factor - margin_px, 0), -(-x1 // downsample_factor) + margin_px)
    return row_slice, column_slice


def plot_fish_qc(fish_id, summary, depths, reference_slices, fish_slices, panel_size=3.5):
    """One row per depth: reference | warped fish | overlay, cropped to the fish's extent.

    Args:
        fish_id (str): Fish identifier.
        summary (dict): From `summarize_fish`.
        depths (list): Reference slice index of each row.
        reference_slices (numpy.ndarray): (depth, y, x) downsampled reference.
        fish_slices (numpy.ndarray): (depth, y, x) downsampled warped fish.
        panel_size (float): Panel width and height, in inches.

    Returns:
        matplotlib.figure.Figure: The figure.
    """
    rows, columns = crop_slices(summary["bounding_box_yxyx"], FISH_FIGURE_DOWNSAMPLE, CROP_MARGIN_PX)
    figure, axes = plt.subplots(len(depths), 3, figsize=(3 * panel_size, len(depths) * panel_size), squeeze=False, layout="compressed")
    for row_axes, depth, reference_slice, fish_slice in zip(axes, depths, reference_slices, fish_slices):
        reference_image = norm01(reference_slice[rows, columns], *REFERENCE_CLIP_PERCENTILES)
        fish_image = scale_fish_image(fish_slice[rows, columns], *FISH_CLIP_PERCENTILES)
        row_axes[0].imshow(reference_image, cmap="gray")
        row_axes[1].imshow(fish_image, cmap="gray")
        row_axes[2].imshow(overlay_rgb(reference_image, fish_image))
        row_axes[0].set_ylabel(f"z={depth} ({depth * Z_SPACING_UM:.0f} µm)")
        for axis in row_axes:
            axis.set_xticks([])
            axis.set_yticks([])
    for axis, title in zip(axes[0], ("reference", fish_id, "overlay (ref magenta, fish green)")):
        axis.set_title(title)
    figure.suptitle(f"{fish_id} → {REFERENCE_NAME}: r = {summary['correlation']:.2f} (pilot {PILOT_CORRELATION:.2f}), "
                    f"coverage {summary['coverage']:.0%}, depth {summary['depth_first_um']:.0f}-{summary['depth_last_um']:.0f} µm")
    return figure


def plot_overview(fish_order, fish_thumbnails, depths, panel_size=2.2):
    """All fish at the same reference depths (overlays), plus a last row with the mean of all fish.

    Args:
        fish_order (list): Fish IDs, one row each, top to bottom.
        fish_thumbnails (dict): fish_id -> (z, y, x) downsampled warped fish.
        depths (list): Reference slice index of each column.
        panel_size (float): Panel width and height, in inches.

    Returns:
        matplotlib.figure.Figure: The figure.
    """
    n_rows = len(fish_order) + 1  # + mean of all fish
    figure, axes = plt.subplots(n_rows, len(depths), figsize=(len(depths) * panel_size, n_rows * panel_size), squeeze=False, layout="compressed")
    for row_axes, fish_id in zip(axes, fish_order):
        for axis, depth in zip(row_axes, depths):
            reference_image = norm01(reference_thumbnail[depth], *REFERENCE_CLIP_PERCENTILES)
            fish_image = scale_fish_image(fish_thumbnails[fish_id][depth], *FISH_CLIP_PERCENTILES)
            axis.imshow(overlay_rgb(reference_image, fish_image))
        row_axes[0].set_ylabel(fish_id)
    for axis, depth in zip(axes[-1], depths):
        mean_image = fish_mean_image([fish_thumbnails[fish_id][depth] for fish_id in fish_order], *FISH_CLIP_PERCENTILES)
        axis.imshow(mean_image, cmap="gray", vmin=0, vmax=1)
    axes[-1][0].set_ylabel("mean of fish")
    for axis, depth in zip(axes[0], depths):
        axis.set_title(f"z={depth} ({depth * Z_SPACING_UM:.0f} µm)")
    for axis in axes.flat:
        axis.set_xticks([])
        axis.set_yticks([])
    figure.suptitle(f"{EXPERIMENT} → {REFERENCE_NAME} (ref magenta, fish green; worst correlation on top)")
    return figure

## 6. Score and plot each fish
One fish at a time (each stack is ~400 MB in memory). Fish without a registration are skipped with a warning.
Each figure shows 5 depths spread over the part of the reference that fish covers.

In [ ]:
summary_rows = []
fish_thumbnails = {}
skipped_fish = {}
for fish in experiment_fish:
    fish_id, fish_folder = fish["fish_id"], fish_folders[fish["fish_id"]]
    if fish_id == REFERENCE_FISH_ID:
        skipped_fish[fish_id] = "is the reference brain"
        continue
    if fish_folder is None:
        skipped_fish[fish_id] = f"no single data folder starting with {fish_id} in {DATA_ROOT}"
        continue
    warped_path = warped_anatomy_path(fish_folder, fish_id, REFERENCE_NAME)
    if not warped_path.exists():
        skipped_fish[fish_id] = f"no warped anatomy at {warped_path}"
        continue
    print(f"=== {fish_id}")
    try:
        warped_stack = read_nrrd_stack(warped_path)
        summary = summarize_fish(warped_stack, reference_stack, Z_SPACING_UM)
    except (OSError, RuntimeError, ValueError) as error:
        skipped_fish[fish_id] = f"could not read or score: {error}"
        continue
    if summary["z_first"] is None:
        skipped_fish[fish_id] = "warped anatomy is empty"
        continue
    depths = spread_depths(summary["z_first"], summary["z_last"], N_DEPTHS)
    fish_slices = downsample_stack(warped_stack[depths], FISH_FIGURE_DOWNSAMPLE)
    fish_thumbnails[fish_id] = downsample_stack(warped_stack, OVERVIEW_DOWNSAMPLE)
    del warped_stack  # only the small copies are kept
    gc.collect()

    reference_slices = downsample_stack(reference_stack[depths], FISH_FIGURE_DOWNSAMPLE)
    figure = plot_fish_qc(fish_id, summary, depths, reference_slices, fish_slices)
    save_figure(figure, QC_DIR / f"{fish_id}_ref_qc.png", QC_DPI)
    summary_rows.append({"fish_id": fish_id, "folder": fish_folder.name, "overview_decision": fish["final_decision"], **summary, "qc_depths": depths})

for fish_id, reason in skipped_fish.items():
    print(f"WARNING {fish_id}: skipped -- {reason}")
print(f"Scored {len(summary_rows)} fish, skipped {len(skipped_fish)}.")

## 7. QC table
Lowest correlation first. Good registrations score about 0.6-0.7 (pilot 0.70); `low_correlation` marks fish under 0.5, look at those first.
Write `ok` / `check` / `bad` and a comment into the CSV's `decision` and `comment` columns: they are kept when this notebook is rerun.

In [ ]:
qc_table = pd.DataFrame(summary_rows).sort_values("correlation", ignore_index=True)
qc_table["low_correlation"] = qc_table["correlation"] < LOW_CORRELATION
qc_table = add_decision_columns(qc_table, read_previous_decisions(TABLE_PATH))
write_qc_table(qc_table, TABLE_PATH, {
    "experiment": EXPERIMENT,
    "reference_name": REFERENCE_NAME,
    "reference_path": reference_anatomy_path(REFERENCE_FOLDER),
    "pilot_correlation": PILOT_CORRELATION,
    "low_correlation_threshold": LOW_CORRELATION,
    "n_depths": N_DEPTHS,
    "skipped_fish": skipped_fish,
})
qc_table[["fish_id", "correlation", "low_correlation", "coverage", "depth_first_um", "depth_last_um", "overview_decision", "decision", "comment"]]

## 8. Overview of the experiment
All fish at the same reference depths, worst correlation on top. A fish that is shifted, flipped or distorted stands
out against the others. The last row averages all fish: sharp structures mean the fish agree, blur means some are off.

In [ ]:
overview_depths = common_depths(list(zip(qc_table["z_first"], qc_table["z_last"])), N_DEPTHS)
overview_figure = plot_overview(qc_table["fish_id"].tolist(), fish_thumbnails, overview_depths)
save_figure(overview_figure, OVERVIEW_PATH, QC_DPI)
print(f"Saved to {QC_DIR}")

## 9. Doubtful fish in Fiji
Open the reference and `<fish>_2P_in_ref.nrrd`, then: **Image → Type → 8-bit** on the fish, **Image → Zoom → View 100%**,
**Image → Color → Merge Channels** (C1 = reference, C2 = fish, *Create composite*), and scroll through z.